Попробуйте написать парсер на Python, который соберет данные с сайта https://www.585zolotoy.ru/ в категории товаров Браслеты – с камнями. Соберите Название товара, Цену после скидок, Цену при оплате online, вес изделия, ссылку на товар. 


In [23]:
import requests

url = "https://www.585zolotoy.ru/catalog/?category=b738558f-be85-4d99-8cdb-397ca740863c"

params = {
    "page": 1,
}

response = requests.get(url, params=params)

In [24]:
response.text[:20]

'<!DOCTYPE html><html'

Как видно выше, запрос вернул html страницу. Значит импортируем BeautifulSoup.

In [25]:
from bs4 import BeautifulSoup
from urllib.parse import urljoin

soup = BeautifulSoup(response.text, "html.parser")

for tag in soup.select('a[href^="/catalog/products/"]'):
    href = tag["href"]
    name = tag.get_text(strip=True)
    full_url = urljoin(
        "https://www.585zolotoy.ru",
        href,
    )

    print(name)
    print(full_url)


Серебряный браслет с гранатом и фианитами
https://www.585zolotoy.ru/catalog/products/3926433/
Серебряный браслет с фианитами
https://www.585zolotoy.ru/catalog/products/4356682/
Серебряный браслет с ониксом
https://www.585zolotoy.ru/catalog/products/1595222/
Серебряный браслет с фианитами
https://www.585zolotoy.ru/catalog/products/1214359/
Золотой браслет с фианитами
https://www.585zolotoy.ru/catalog/products/9000647/
Серебряный браслет с топазами Sky и фианитами
https://www.585zolotoy.ru/catalog/products/2003129/
Серебряный браслет с ониксом
https://www.585zolotoy.ru/catalog/products/1914373/
Серебряный браслет с перламутром
https://www.585zolotoy.ru/catalog/products/1694729/
Серебряный браслет с фианитами
https://www.585zolotoy.ru/catalog/products/2591769/
Золотой браслет с фианитами
https://www.585zolotoy.ru/catalog/products/9003392/
Золотой браслет с бриллиантами
https://www.585zolotoy.ru/catalog/products/1053719/
Серебряный браслет с фианитами
https://www.585zolotoy.ru/catalog/prod

Так мы будем проходиться по всем страницам и по всем page.

In [29]:
url_page = "https://www.585zolotoy.ru/catalog/products/9000103/"
params_page = {

}
response_page = requests.get(url_page)

with open("jel.html", "w") as file:
    file.write(response_page.text)


In [30]:
from IPython.display import JSON, display
import json

soup = BeautifulSoup(
    response_page.text, "html.parser"
)

script_tag = soup.find(
    "script",
    id="__NUXT_DATA__"
)

raw_json = script_tag.get_text()

with open("data.html", "w") as file:
    file.write(raw_json)

In [45]:
import json
import re
import pandas as pd

from pathlib import Path
from bs4 import BeautifulSoup


def parse_product(html):
    soup = BeautifulSoup(html, "html.parser")

    # Название
    name = soup.find("h1").get_text(strip=True)

    # Цена после скидок
    price_text = soup.select_one(
        "#product-price .text-title-2-medium"
    ).get_text(strip=True)

    price = int(re.sub(r"\D", "", price_text))

    # Вес
    weight_label = soup.find(
        "div",
        string=lambda text:
          text and text.strip() in ["Вес", "Вес изделия"]
    )

    weight_text = None
    weight = None

    if weight_label is not None:
        weight_value = weight_label.find_next_sibling(
            "div"
        )
        if weight_value is not None:
            weight_text = weight_value.get_text(" ", strip=True)

        weight_match = re.search(
            r"\d+(?:[.,]\d+)?",
            weight_text
        )

        if weight_match is not None:
            weight = float(
                weight_match
                .group()
                .replace(",", ".")
            )
    

    # Ссылка на товар
    link = soup.find("link", rel="canonical")["href"]

    # Получаем процент скидки при оплате online
    nuxt_tag = soup.find("script", id="__NUXT_DATA__")
    nuxt_data = json.loads(nuxt_tag.get_text())

    settings = next(
        item
        for item in nuxt_data
        if isinstance(item, dict)
        and "online_discount_percent" in item
    )

    discount_reference = settings["online_discount_percent"]
    online_discount = nuxt_data[discount_reference]

    online_price = round(
        price * (1 - online_discount / 100)
    )

    return {
        "Название товара": name,
        "Цена после скидок": price,
        "Цена при оплате online": online_price,
        "Вес, г": weight,
        "Ссылка": link
    }

In [46]:
html = Path("jel.html").read_text(encoding="utf-8")

product = parse_product(html)
product

{'Название товара': 'Серебряный браслет с фианитами',
 'Цена после скидок': 10796,
 'Цена при оплате online': 10256,
 'Вес, г': 17.0,
 'Ссылка': 'https://www.585zolotoy.ru/catalog/products/1152175/'}

In [47]:
import requests

url = "https://www.585zolotoy.ru/catalog/braslety-iz-kamney/"

products = []

for page in range(1, 7):
    params = {
        "page": page
    }

    response = requests.get(
        url,
        params=params,
        timeout=30
    )

    soup = BeautifulSoup(response.text, "html.parser")

    for tag in soup.select('a[href^="/catalog/products/"]'):
        href = tag["href"]
        name = tag.get_text(strip=True)
        full_url = urljoin(
            "https://www.585zolotoy.ru",
            href,
        )

        url_page = full_url
  
        response_page = requests.get(url_page, timeout=30)

        html = response_page.text

        product = parse_product(html)
        products.append(product)



    

ConnectTimeout: HTTPSConnectionPool(host='www.585zolotoy.ru', port=443): Max retries exceeded with url: /catalog/products/9018359/ (Caused by ConnectTimeoutError(<HTTPSConnection(host='www.585zolotoy.ru', port=443) at 0x11542b9d0>, 'Connection to www.585zolotoy.ru timed out. (connect timeout=30)'))

In [ ]:
product_df = pd.DataFrame(products)

In [ ]:
product_df.to_csv(
    "products.csv",
    index=False,
    encoding="utf-8-sig"
)

In [ ]:
check_df = pd.read_csv(
    "products.csv",
    encoding="utf-8-sig",
    sep=";"
)

check_df.head()

,"Название товара,Цена после скидок,Цена при оплате online,""Вес, г"",Ссылка"
0,"Серебряный браслет с фианитами,2084,1980,1.76,..."
1,"Серебряный браслет с фианитами,2084,1980,1.76,..."
2,"Серебряный браслет с фианитами,2084,1980,1.76,..."
3,"Серебряный браслет с фианитами,2084,1980,1.76,..."
4,"Серебряный браслет с фианитами,2084,1980,1.76,..."
